Notes will bounce back and forth between here and the Obsidian notebook.
let's get started with some tokenization, start by grabbing data;

In [1]:
from fastai.text.all import *
path = untar_data(URLs.IMDB)

In [2]:
files = get_text_files(path, folders = ['train', 'test', 'unsup'])

In [3]:
#Here's one review that we'll tokenize:
txt = files[0].open().read(); txt[:75]

'Lets see it starts out with this whole "Troy-ish" Battle directly lifted ou'

We'll use fastai's coll_repr(collection, n) function to display the results. This displays the first n items of collection, along with the full size—it's what L uses by default. Note that fastai's tokenizers take a collection of documents to tokenize, so we have to wrap txt in a list:

In [4]:
#WordTokenizer() should call the most recent fast ai tokenizer
spacy = WordTokenizer()
toks = first(spacy([txt]))
print(coll_repr(toks, 30))

(#157) ['Lets','see','it','starts','out','with','this','whole','"','Troy','-','ish','"','Battle','directly','lifted','out','n','copied','or','even','remade','to','an','extent','..','<br','/><br','/>then','we'...]


In [5]:
#By default it is spacy, you can check which one it is like this.
print(type(spacy))

<class 'fastai.text.core.SpacyTokenizer'>


In [6]:
#fastai adds additional functionality to tokenization process
tkn = Tokenizer(spacy)
print(coll_repr(tkn(txt), 31))

(#164) ['xxbos','xxmaj','lets','see','it','starts','out','with','this','whole','"','troy','-','ish','"','xxmaj','battle','directly','lifted','out','n','copied','or','even','remade','to','an','extent','..','\n\n','then'...]


tokens with xx in front represent non english words or punctuation
xxbos is beginning of stream, or beginning of document.
xxmaj means next word begins with capital
xxunk means there is a word that is unknown aka not in our vocabulary.
These don't come from spaCy, fastai adds them by default.
"For instance, the rules will replace a sequence of four exclamation points with a special repeated character token, followed by the number four, and then a single exclamation point."

In [7]:
#check the rules that were used:
defaults.text_proc_rules

[<function fastai.text.core.fix_html(x)>,
 <function fastai.text.core.replace_rep(t)>,
 <function fastai.text.core.replace_wrep(t)>,
 <function fastai.text.core.spec_add_spaces(t)>,
 <function fastai.text.core.rm_useless_spaces(t)>,
 <function fastai.text.core.replace_all_caps(t)>,
 <function fastai.text.core.replace_maj(t)>,
 <function fastai.text.core.lowercase(t, add_bos=True, add_eos=False)>]

In [9]:
## and look up the code for each
??replace_rep

Signature: replace_rep(t)
Source:   
def replace_rep(t):
    "Replace repetitions at the character level: cccc -- TK_REP 4 c"
    def _replace_rep(m):
        c,cc = m.groups()
        return f' {TK_REP} {len(cc)+1} {c} '
    return _re_rep.sub(_replace_rep, t)
File:      ~/miniforge3/envs/fastai/lib/python3.12/site-packages/fastai/text/core.py
Type:      function

Here is a brief summary of what each does:

fix_html:: Replaces special HTML characters with a readable version (IMDb reviews have quite a few of these)<br>
replace_rep:: Replaces any character repeated three times or more with a special token for repetition (xxrep), the number of times it's repeated, then the character <br>
replace_wrep:: Replaces any word repeated three times or more with a special token for word repetition (xxwrep), the number of times it's repeated, then the word <br>
spec_add_spaces:: Adds spaces around / and # <br>
rm_useless_spaces:: Removes all repetitions of the space character <br>
replace_all_caps:: Lowercases a word written in all caps and adds a special token for all caps (xxup) in front of it <br>
replace_maj:: Lowercases a capitalized word and adds a special token for capitalized (xxmaj) in front of it <br>
lowercase:: Lowercases all text and adds a special token at the beginning (xxbos) and/or the end (xxeos)<br>

In [10]:
#example:
coll_repr(tkn('&copy;   Fast.ai www.fast.ai/INDEX'), 31)

"(#11) ['xxbos','©','xxmaj','fast.ai','xxrep','3','w','.fast.ai','/','xxup','index']"

In [11]:
#grabbing first 2000 reviews:
txts = L(o.open().read() for o in files[:2000])

In [13]:
#instantiate subword tokenizer with vocab size we want
#train it with .setup, passing the texts.
def subword(sz):
    sp = SubwordTokenizer(vocab_sz=sz)
    sp.setup(txts)
    return ' '.join(first(sp([txt]))[:40])
#note this is hard coded to use the txts we set up in the previous cell

In [14]:
subword(1000)

sentencepiece_trainer.cc(178) LOG(INFO) Running command: --input=tmp/texts.out --vocab_size=1000 --model_prefix=tmp/spm --character_coverage=0.99999 --model_type=unigram --unk_id=9 --pad_id=-1 --bos_id=-1 --eos_id=-1 --minloglevel=2 --user_defined_symbols=▁xxunk,▁xxpad,▁xxbos,▁xxeos,▁xxfld,▁xxrep,▁xxwrep,▁xxup,▁xxmaj --hard_vocab_limit=false


'▁Le t s ▁see ▁it ▁start s ▁out ▁with ▁this ▁whole ▁" T ro y - ish " ▁B at t le ▁direct ly ▁li f t ed ▁out ▁ n ▁co p i ed ▁or ▁even ▁re ma de'

In [16]:
#'_' is a space in original text, spaces between vocab words.
#if we make the vocab size small we'll have smaller tokens and need more tokens to make a sentence:
subword(200)


'▁ L e t s ▁s e e ▁it ▁s t ar t s ▁ o u t ▁with ▁this ▁w h o le ▁ " T ro y - i s h " ▁ B a t t le'

In [17]:
#if we make our vocab much larger we'll get bigger words, actual english words:
subword(10000)

'▁Let s ▁see ▁it ▁starts ▁out ▁with ▁this ▁whole ▁" T ro y - ish " ▁Battle ▁direct ly ▁lift ed ▁out ▁n ▁copie d ▁or ▁even ▁remade ▁to ▁an ▁extent . . < br ▁/> < br ▁/> then'

in short, larger vocab = fewer tokens/sentence, faster training, less memory
downsides are larger embedding matrices which need more data to learn<br>
Now we need to numericalize our text: mapping tokens to integers.

In [18]:
#Here's our tokenized text from above:
toks = tkn(txt)
print(coll_repr(tkn(txt), 31))

(#164) ['xxbos','xxmaj','lets','see','it','starts','out','with','this','whole','"','troy','-','ish','"','xxmaj','battle','directly','lifted','out','n','copied','or','even','remade','to','an','extent','..','\n\n','then'...]


Numericalizer needs to be initialized with setup.

In [21]:
#remember .map(x) performs function x on everything in the input collation.
#tkn is the spacy tokenizer we made way further up
#using a subset of our txts because this can take a long time
toks200 = txts[:200].map(tkn)
toks200[0]


(#164) ['xxbos','xxmaj','lets','see','it','starts','out','with','this','whole','"','troy','-','ish','"','xxmaj','battle','directly','lifted','out'...]

In [20]:
toks200[1]

(#226) ['xxbos','i','can','not','express','in','words','how','many','different','styles','of','film','making','xxmaj','wes','xxmaj','anderson','combined','into'...]

In [22]:
#list of lists of tokens being sent to setup
num = Numericalize()
num.setup(toks200)
coll_repr(num.vocab,20)

"(#2248) ['xxunk','xxpad','xxbos','xxeos','xxfld','xxrep','xxwrep','xxup','xxmaj','the',',','.','a','and','of','to','is','it','in','i'...]"

Notice that special characters appear first, followed by most frequent other tokens <br>
Defaults to Numericalize:<br>
min_freq=3,max_vocab=60000<br>
If frequency less than min, replace with UNK<br>
Also will replace all other than 6000 most common with UNK<br>
Helps prevent overly large embedding matrix (slows training, uses memory)<br>
Minimum frequency also helps prevent having too little data to train useful representations of the word.<br>
You can pass in a list of words as the vocab parameter to set a custom vocab.

In [23]:
#now we can call our numericalizer num as a function
nums = num(toks)[:20];
nums

TensorText([   2,    8,  907,   86,   17,  455,   61,   29,   21,  196,   22,
               0,   25, 1644,   22,    8,  792, 1062, 1324,   61])

In [24]:
#we can check that these integers actually map back to the vocabulary:
' '.join(num.vocab[o] for o in nums)

'xxbos xxmaj lets see it starts out with this whole " xxunk - ish " xxmaj battle directly lifted out'

In [25]:
#LMDataLoader will create batches for us, read the obsidian notebook for details on that
#first numericalize our tokenized text:
nums200 = toks200.map(num)

In [26]:
#then pass to LMDataLoader
dl = LMDataLoader(nums200)

In [27]:
#let's see if it worked
x,y = first(dl)
x.shape,y.shape

(torch.Size([64, 72]), torch.Size([64, 72]))

In [28]:
#Looking at first row of independant var, should be start of first text:
' '.join(num.vocab[o] for o in x[0][:20])

'xxbos xxmaj lets see it starts out with this whole " xxunk - ish " xxmaj battle directly lifted out'

In [29]:
#dependent variable offset by 1 token
' '.join(num.vocab[o] for o in y[0][:20])

'xxmaj lets see it starts out with this whole " xxunk - ish " xxmaj battle directly lifted out xxunk'

Preprocessing complete! fastai actually just does all this when you pass TextBlock to DataBlock, but this was a good look under the hood. We can pass these options to DataBlock.<br>
Let's create a language model using fastai's defaults:

In [30]:
get_imdb = partial(get_text_files, folders=['train', 'test', 'unsup'])

dls_lm = DataBlock(
    blocks=TextBlock.from_folder(path, is_lm=True),
    get_items=get_imdb, splitter=RandomSplitter(0.1)
).dataloaders(path, path=path, bs=128, seq_len=80)

In [31]:
dls_lm.show_batch(max_n=2)

xxbos xxmaj it seems natural that at the very start of xxmaj chaplin 's career he should make a movie in which he plays a grossly unqualified boxer who succeeds by sneaking horse shoes into his gloves . i only wonder why it did n't happen even earlier . xxmaj he passes a sign saying that sparring partners are wanted , people who know how to take a punch , and heads in . xxmaj we see him grow increasingly
love xxmaj actually with xxup xxunk ! xxbos xxmaj skullduggery , skullduggery , ah , how you have hurt so many with your cheesy - ness . xxmaj yet , i love this movie , not as much as ' big xxmaj trouble in xxmaj little xxmaj china ' or ' trekkies , ' but i love ' skullduggery ' all the same . xxmaj the entire movie reeks of low budget " student film " and there are more
xxmaj it seems natural that at the very start of xxmaj chaplin 's career he should make a movie in which he plays a grossly unqualified boxer who succeeds by sneaking horse shoes into his gloves . i only wonder why it 

In [32]:
#making learner using AWD_LSTM architecture
learn = language_model_learner(
    dls_lm, AWD_LSTM, drop_mult=0.3, 
    metrics=[accuracy, Perplexity()]).to_fp16()

/home/puhi/miniforge3/envs/fastai/lib/python3.12/site-packages/fastai/text/learner.py:149: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  wgts = torch.load(wgts_fname, map_lo

In [33]:
#full epoch takes too long so use fit_one_cycle.
learn.fit_one_cycle(1, 2e-2)

/home/puhi/miniforge3/envs/fastai/lib/python3.12/site-packages/fastai/callback/fp16.py:47: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  self.autocast,self.learn.scaler,self.scales = autocast(dtype=dtype),GradScaler(**self.kwargs),L()
/home/puhi/miniforge3/envs/fastai/lib/python3.12/site-packages/fastai/callback/fp16.py:47: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.autocast,self.learn.scaler,self.scales = autocast(dtype=dtype),GradScaler(**self.kwargs),L()


epoch,train_loss,valid_loss,accuracy,perplexity,time
0,3.999202,3.896728,0.300944,49.241070,14:01


In [34]:
#fit one cycle calls freeze after, we can save/load our model:
#learner needs to be set up same way to load
learn.save('1epoch')

Path('/home/puhi/.fastai/data/imdb/models/1epoch.pth')

In [35]:
learn = learn.load('1epoch')

/home/puhi/miniforge3/envs/fastai/lib/python3.12/site-packages/fastai/text/learner.py:92: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(file, map_location

In [36]:
learn.unfreeze()
learn.fit_one_cycle(10, 2e-3)

/home/puhi/miniforge3/envs/fastai/lib/python3.12/site-packages/fastai/callback/fp16.py:47: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  self.autocast,self.learn.scaler,self.scales = autocast(dtype=dtype),GradScaler(**self.kwargs),L()
/home/puhi/miniforge3/envs/fastai/lib/python3.12/site-packages/fastai/callback/fp16.py:47: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.autocast,self.learn.scaler,self.scales = autocast(dtype=dtype),GradScaler(**self.kwargs),L()


epoch,train_loss,valid_loss,accuracy,perplexity,time
0,3.746275,3.754290,0.317587,42.703873,14:22
1,3.707637,3.696738,0.323748,40.315590,13:11
2,3.641321,3.649161,0.328916,38.442402,13:12
3,3.577742,3.614259,0.333474,37.123844,13:05
4,3.492035,3.594327,0.336080,36.391201,13:09
5,3.430095,3.577314,0.338293,35.777309,13:44
6,3.370661,3.569054,0.340053,35.483006,13:51
7,3.294207,3.565381,0.340991,35.352932,13:48
8,3.242981,3.568449,0.341046,35.461552,13:55
9,3.217170,3.572745,0.340827,35.614216,11:35


In [37]:
# 2 hours, 8 minutes, and 52 seconds
# I'm interested to see how Transformers compare to this.
#save the encoder:
learn.save_encoder('finetuned')

In [38]:
#since we have an updated language model we can generate new reviews;

In [ ]:
TEXT = "I liked this movie because"
N_WORDS = 40
N_SENTENCES = 2
preds = [learn.predict(TEXT, N_WORDS, temperature=0.75) 
         for _ in range(N_SENTENCES)]
#temperature is like a randomness input, 1.0 will give very random, 0 always picks most likely word.

In [40]:
print("\n".join(preds))

i liked this movie because it was so good . And i thought it was a good movie . i think it was very enjoyable as well . i would recommend it to anyone looking for a good laugh . i think the movie
i liked this movie because it was not only the first Peanuts movie but also the best Peanuts ever ! i thought Peanuts was a great comedy of the week and it was a great fun family film . The characters


Now let's check out how we can use our encoder to train the sentiment classifier.

In [41]:
#two important notes, this one has no is_lm=True (lm = language model)
#we eed to pass our vocab so the indexes of our embeddings make sense
dls_clas = DataBlock(
    blocks=(TextBlock.from_folder(path, vocab=dls_lm.vocab),CategoryBlock),
    get_y = parent_label,
    get_items=partial(get_text_files, folders=['train', 'test']),
    splitter=GrandparentSplitter(valid_name='test')
).dataloaders(path, path=path, bs=128, seq_len=72)

In [42]:
dls_clas.show_batch(max_n=3)

,text,category
0,"xxbos xxmaj match 1 : xxmaj tag xxmaj team xxmaj table xxmaj match xxmaj bubba xxmaj ray and xxmaj spike xxmaj dudley vs xxmaj eddie xxmaj guerrero and xxmaj chris xxmaj benoit xxmaj bubba xxmaj ray and xxmaj spike xxmaj dudley started things off with a xxmaj tag xxmaj team xxmaj table xxmaj match against xxmaj eddie xxmaj guerrero and xxmaj chris xxmaj benoit . xxmaj according to the rules of the match , both opponents have to go through tables in order to get the win . xxmaj benoit and xxmaj guerrero heated up early on by taking turns hammering first xxmaj spike and then xxmaj bubba xxmaj ray . a xxmaj german xxunk by xxmaj benoit to xxmaj bubba took the wind out of the xxmaj dudley brother . xxmaj spike tried to help his brother , but the referee restrained him while xxmaj benoit and xxmaj guerrero",pos
1,"xxbos * ! ! - xxup spoilers - ! ! * \n\n xxmaj before i begin this , let me say that i have had both the advantages of seeing this movie on the big screen and of having seen the "" authorized xxmaj version "" of this movie , remade by xxmaj stephen xxmaj king , himself , in 1997 . \n\n xxmaj both advantages made me appreciate this version of "" the xxmaj shining , "" all the more . \n\n xxmaj also , let me say that xxmaj i 've read xxmaj mr . xxmaj king 's book , "" the xxmaj shining "" on many occasions over the years , and while i love the book and am a huge fan of his work , xxmaj stanley xxmaj kubrick 's retelling of this story is far more compelling … and xxup scary . \n\n xxmaj kubrick",pos
2,"xxbos xxmaj raising xxmaj victor xxmaj vargas : a xxmaj review \n\n xxmaj you know , xxmaj raising xxmaj victor xxmaj vargas is like sticking your hands into a big , steaming bowl of oatmeal . xxmaj it 's warm and gooey , but you 're not sure if it feels right . xxmaj try as i might , no matter how warm and gooey xxmaj raising xxmaj victor xxmaj vargas became i was always aware that something did n't quite feel right . xxmaj victor xxmaj vargas suffers from a certain overconfidence on the director 's part . xxmaj apparently , the director thought that the ethnic backdrop of a xxmaj latino family on the lower east side , and an idyllic storyline would make the film critic proof . xxmaj he was right , but it did n't fool me . xxmaj raising xxmaj victor xxmaj vargas is",neg


In [43]:
#Let's take a quick look at our example sizes:
nums_samp = toks200[:10].map(num)
nums_samp.map(len)

(#10) [164,226,323,518,501,199,198,234,300,1204]

In [44]:
#definitely some different lengths, check the notes for how we minibatch this.
#DataBlock takes care of that for us.
#create the model:
learn = text_classifier_learner(dls_clas, AWD_LSTM, drop_mult=0.5, 
                                metrics=accuracy).to_fp16()

/home/puhi/miniforge3/envs/fastai/lib/python3.12/site-packages/fastai/text/learner.py:149: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  wgts = torch.load(wgts_fname, map_lo

In [45]:
#load our encoder from before, then we can train!:
learn = learn.load_encoder('finetuned')

/home/puhi/miniforge3/envs/fastai/lib/python3.12/site-packages/fastai/text/learner.py:135: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  wgts = torch.load(join_path_file(fil

In [46]:
#starting with one cycle
#we're going to slowly unfreeze layers
learn.fit_one_cycle(1, 2e-2)


/home/puhi/miniforge3/envs/fastai/lib/python3.12/site-packages/fastai/callback/fp16.py:47: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  self.autocast,self.learn.scaler,self.scales = autocast(dtype=dtype),GradScaler(**self.kwargs),L()
/home/puhi/miniforge3/envs/fastai/lib/python3.12/site-packages/fastai/callback/fp16.py:47: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.autocast,self.learn.scaler,self.scales = autocast(dtype=dtype),GradScaler(**self.kwargs),L()


epoch,train_loss,valid_loss,accuracy,time
0,0.250357,0.181407,0.930520,00:32


In [47]:
#slice here is enforcing different learning rates on different layers
learn.freeze_to(-2)
learn.fit_one_cycle(1, slice(1e-2/(2.6**4),1e-2))

/home/puhi/miniforge3/envs/fastai/lib/python3.12/site-packages/fastai/callback/fp16.py:47: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  self.autocast,self.learn.scaler,self.scales = autocast(dtype=dtype),GradScaler(**self.kwargs),L()
/home/puhi/miniforge3/envs/fastai/lib/python3.12/site-packages/fastai/callback/fp16.py:47: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.autocast,self.learn.scaler,self.scales = autocast(dtype=dtype),GradScaler(**self.kwargs),L()


epoch,train_loss,valid_loss,accuracy,time
0,0.218988,0.164325,0.936440,00:43


In [48]:
learn.freeze_to(-3)
learn.fit_one_cycle(1, slice(5e-3/(2.6**4),5e-3))

/home/puhi/miniforge3/envs/fastai/lib/python3.12/site-packages/fastai/callback/fp16.py:47: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  self.autocast,self.learn.scaler,self.scales = autocast(dtype=dtype),GradScaler(**self.kwargs),L()
/home/puhi/miniforge3/envs/fastai/lib/python3.12/site-packages/fastai/callback/fp16.py:47: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.autocast,self.learn.scaler,self.scales = autocast(dtype=dtype),GradScaler(**self.kwargs),L()


epoch,train_loss,valid_loss,accuracy,time
0,0.192188,0.150452,0.943200,00:57


In [49]:
learn.unfreeze()
learn.fit_one_cycle(2, slice(1e-3/(2.6**4),1e-3))

/home/puhi/miniforge3/envs/fastai/lib/python3.12/site-packages/fastai/callback/fp16.py:47: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  self.autocast,self.learn.scaler,self.scales = autocast(dtype=dtype),GradScaler(**self.kwargs),L()
/home/puhi/miniforge3/envs/fastai/lib/python3.12/site-packages/fastai/callback/fp16.py:47: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.autocast,self.learn.scaler,self.scales = autocast(dtype=dtype),GradScaler(**self.kwargs),L()


epoch,train_loss,valid_loss,accuracy,time
0,0.163540,0.148589,0.945800,00:49
1,0.151321,0.149116,0.947000,00:47


In [54]:
#classify a test review
test_text = " I’ll have two number 9s, a number 9 large, a number 6 with extra dip, a number 7, two number 45s, one with cheese, and a large soda."
prediction = learn.predict(test_text)

predicted_category = prediction[0]
predicted_category_index = prediction[1]
probabilities = prediction[2]

print(f"category: {predicted_category}")
print(f"index: {predicted_category_index}")
print(f"probs: {probabilities}")

/home/puhi/miniforge3/envs/fastai/lib/python3.12/site-packages/fastai/callback/fp16.py:47: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  self.autocast,self.learn.scaler,self.scales = autocast(dtype=dtype),GradScaler(**self.kwargs),L()
/home/puhi/miniforge3/envs/fastai/lib/python3.12/site-packages/fastai/callback/fp16.py:47: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.autocast,self.learn.scaler,self.scales = autocast(dtype=dtype),GradScaler(**self.kwargs),L()


category: pos
index: 1
probs: tensor([0.1125, 0.8875])
